In [1]:
import pickle
import os
import pandas as pd
import pickle
import numpy as np
import ast

In [19]:
BIOS_LABELS = "data/BIOS_LABELS_v2.csv" # this contains all of our changes without the sensitive data, but ids matching the BIOS dataset
BIOS_RAW = "../../data/biosbias2023/BIOS.pkl" # path to the downloaded BIOS.pkl

reviewed_classes = ['architect', 'surgeon', 'dentist', 'teacher', 'psychologist', 'nurse', 'photographer', 'physician', 'attorney', 'journalist', 'software_enginner', 'model']

In [9]:
# get raw data from crawler
with (open(BIOS_RAW, "rb")) as openfile:
    raw_data = pickle.load(openfile)
df_raw = pd.DataFrame(raw_data)
df_raw['index'] = df_raw.index
df_raw = df_raw.rename(columns={'raw_title': 'auto_raw_title', 'title': 'auto_title'})
df_raw

,path,raw,name,auto_raw_title,gender,start_pos,auto_title,URI,bio,index
0,crawl-data/CC-MAIN-2013-20/segments/1368696381...,* Nora Fisher Onar is an assistant professor o...,"(Nora, Fisher, Onar)",assistant professor,F,109,professor,http://acturca.wordpress.com/2012/04/13/turkey...,_ is also a Ronald D. Asmus Policy Entrepreneu...,0
1,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Brent Wagner is a certified public accountant ...,"(Brent, , Wagner)",certified public accountant,M,124,accountant,http://www.forwardtheater.com/about/board,_ is a member of the AICPA and WICPA. _ gradua...,1
2,crawl-data/CC-MAIN-2014-41/segments/1410657102...,Dr. Shawn Zelig Aster is an assistant professo...,"(Shawn, Zelig, Aster)",assistant professor,M,79,professor,http://yuspeakers.org/speakers/,Dr. _ has held teaching and research positions...,2
3,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Brittany McCandless is a journalist with a pri...,"(Brittany, , McCandless)",journalist,F,74,journalist,http://morevoices.tumblr.com/,_ currently works on CNN’s newest primetime sh...,3
4,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Arturo Toledo is an Architect. He runs a bouti...,"(Arturo, , Toledo)",Architect,M,30,architect,http://diggthedrazen.com/author/diggthedrazen/,_ runs a boutique design studio attending clie...,4
...,...,...,...,...,...,...,...,...,...,...
397902,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Rebecca Wexler is a documentary filmmaker; co-...,"(Rebecca, , Wexler)",filmmaker,F,322,filmmaker,https://www.thecommonsjournal.org/articles/10....,_ recently returned from working as a Fulbrigh...,397902
397903,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Fernando Quesada is an Attorney at-law admitte...,"(Fernando, , Quesada)",Attorney,M,66,attorney,https://dentons.lopez-velarde.com/en/fernando-...,_ focuses _ practice in all forms of Energy (i...,397903
397904,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Jeffrey Kucik is an assistant professor in the...,"(Jeffrey, , Kucik)",assistant professor,M,115,professor,http://www.latimes.com/opinion/op-ed/la-oe-kuc...,_ maintains the trade policy site www.trademon...,397904
397905,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Leslie E. Seawright is an Assistant Professor ...,"(Leslie, E, Seawright)",Assistant Professor,F,107,professor,https://www.abebooks.com/9781443863339/Going-G...,"_ teaches courses in Technical Writing, Compos...",397905


In [10]:
# get labels
df_label = pd.read_csv(BIOS_LABELS, sep='\t')
df_label

,index,path,gender,start_pos,URI,auto_raw_title,auto_title,review,raw_titles,titles,valid,style_valid
0,0,crawl-data/CC-MAIN-2013-20/segments/1368696381...,F,109,http://acturca.wordpress.com/2012/04/13/turkey...,assistant professor,professor,0.0,NaN,NaN,-1.0,-1.0
1,1,crawl-data/CC-MAIN-2013-20/segments/1368696381...,M,124,http://www.forwardtheater.com/about/board,certified public accountant,accountant,0.0,NaN,NaN,-1.0,-1.0
2,2,crawl-data/CC-MAIN-2014-41/segments/1410657102...,M,79,http://yuspeakers.org/speakers/,assistant professor,professor,0.0,NaN,NaN,-1.0,-1.0
3,3,crawl-data/CC-MAIN-2013-20/segments/1368696381...,F,74,http://morevoices.tumblr.com/,journalist,journalist,0.0,NaN,NaN,-1.0,-1.0
4,4,crawl-data/CC-MAIN-2013-20/segments/1368696381...,M,30,http://diggthedrazen.com/author/diggthedrazen/,Architect,architect,0.0,NaN,NaN,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
397902,397902,crawl-data/CC-MAIN-2018-43/segments/1539583519...,F,322,https://www.thecommonsjournal.org/articles/10....,filmmaker,filmmaker,0.0,NaN,NaN,-1.0,-1.0
397903,397903,crawl-data/CC-MAIN-2018-43/segments/1539583519...,M,66,https://dentons.lopez-velarde.com/en/fernando-...,Attorney,attorney,0.0,NaN,NaN,-1.0,-1.0
397904,397904,crawl-data/CC-MAIN-2018-43/segments/1539583519...,M,115,http://www.latimes.com/opinion/op-ed/la-oe-kuc...,assistant professor,professor,0.0,NaN,NaN,-1.0,-1.0
397905,397905,crawl-data/CC-MAIN-2018-43/segments/1539583519...,F,107,https://www.abebooks.com/9781443863339/Going-G...,Assistant Professor,professor,0.0,NaN,NaN,-1.0,-1.0


### Merges texts and names from crawled dataset into the label patch

In [12]:
COLUMNS_TO_MERGE = ['index', 'review', 'raw_titles', 'titles', 'valid']

merged_df = pd.merge(
    df_raw,
    df_label[COLUMNS_TO_MERGE],
    on='index',
    how='left'
)

In [13]:
def parse_titles(value, with_brackets=True):
    """Convert a cell of the 'titles' column to a list of strings."""
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        value = value.strip()
        if not value:                      # empty string -> no labels
            return []
        try:
            parsed = ast.literal_eval(value)   # handles "['a', 'b']"
            if isinstance(parsed, list):
                return [str(x) for x in parsed]
        except (ValueError, SyntaxError):
            pass
        # fallback: plain string with a delimiter, e.g. "spam, ham"
        return [part.strip() for part in value.split(',') if part.strip()]
    return []  

In [14]:
merged_df['raw_titles'] = merged_df['raw_titles'].apply(parse_titles)
merged_df['titles'] = merged_df['titles'].apply(parse_titles)
merged_df

,path,raw,name,auto_raw_title,gender,start_pos,auto_title,URI,bio,index,review,raw_titles,titles,valid
0,crawl-data/CC-MAIN-2013-20/segments/1368696381...,* Nora Fisher Onar is an assistant professor o...,"(Nora, Fisher, Onar)",assistant professor,F,109,professor,http://acturca.wordpress.com/2012/04/13/turkey...,_ is also a Ronald D. Asmus Policy Entrepreneu...,0,0.0,[],[],-1.0
1,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Brent Wagner is a certified public accountant ...,"(Brent, , Wagner)",certified public accountant,M,124,accountant,http://www.forwardtheater.com/about/board,_ is a member of the AICPA and WICPA. _ gradua...,1,0.0,[],[],-1.0
2,crawl-data/CC-MAIN-2014-41/segments/1410657102...,Dr. Shawn Zelig Aster is an assistant professo...,"(Shawn, Zelig, Aster)",assistant professor,M,79,professor,http://yuspeakers.org/speakers/,Dr. _ has held teaching and research positions...,2,0.0,[],[],-1.0
3,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Brittany McCandless is a journalist with a pri...,"(Brittany, , McCandless)",journalist,F,74,journalist,http://morevoices.tumblr.com/,_ currently works on CNN’s newest primetime sh...,3,0.0,[],[],-1.0
4,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Arturo Toledo is an Architect. He runs a bouti...,"(Arturo, , Toledo)",Architect,M,30,architect,http://diggthedrazen.com/author/diggthedrazen/,_ runs a boutique design studio attending clie...,4,0.0,[],[],-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397902,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Rebecca Wexler is a documentary filmmaker; co-...,"(Rebecca, , Wexler)",filmmaker,F,322,filmmaker,https://www.thecommonsjournal.org/articles/10....,_ recently returned from working as a Fulbrigh...,397902,0.0,[],[],-1.0
397903,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Fernando Quesada is an Attorney at-law admitte...,"(Fernando, , Quesada)",Attorney,M,66,attorney,https://dentons.lopez-velarde.com/en/fernando-...,_ focuses _ practice in all forms of Energy (i...,397903,0.0,[],[],-1.0
397904,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Jeffrey Kucik is an assistant professor in the...,"(Jeffrey, , Kucik)",assistant professor,M,115,professor,http://www.latimes.com/opinion/op-ed/la-oe-kuc...,_ maintains the trade policy site www.trademon...,397904,0.0,[],[],-1.0
397905,crawl-data/CC-MAIN-2018-43/segments/1539583519...,Leslie E. Seawright is an Assistant Professor ...,"(Leslie, E, Seawright)",Assistant Professor,F,107,professor,https://www.abebooks.com/9781443863339/Going-G...,"_ teaches courses in Technical Writing, Compos...",397905,0.0,[],[],-1.0


In [15]:
def filter_dataset(df: pd.DataFrame, classes: list, single_label=True, review_only=True, valid_only=True, remove_other_classes=True):
    if review_only:
        df = df[df['review'] == 1]
    if valid_only:
        df = df[df['valid'] == 1]

    if single_label:
        # Keep only rows whose 'titles' list has exactly one element
        df = df[df['titles'].apply(lambda t: isinstance(t, list) and len(t) == 1)]

        # Extract the single label as an int index into `classes`;
        # -1 means "class not in `classes`"
        def to_single_label(titles):
            title = titles[0]
            return classes.index(title) if title in classes else -1

        df = df.copy()
        df['label'] = df['titles'].apply(to_single_label)

        if remove_other_classes:
            df = df[df['label'] != -1]
    else:
        # One binary column per class (multi-label), 1 if the title is present
        df = df.copy()
        for cls in classes:
            df[f'label_{cls}'] = df['titles'].apply(
                lambda t: int(cls in t) if isinstance(t, list) else 0
            )

        if remove_other_classes:
            # Drop rows containing any title that is not in `classes`
            classes_set = set(classes)
            df = df[df['titles'].apply(
                lambda t: isinstance(t, list) and all(x in classes_set for x in t)
            )]

    return df

In [16]:
# only reviewed samples, as multi-label
df = filter_dataset(merged_df, classes=reviewed_classes, single_label=False, review_only=True, valid_only=False, remove_other_classes=False)
df

,path,raw,name,auto_raw_title,gender,start_pos,auto_title,URI,bio,index,...,label_dentist,label_teacher,label_psychologist,label_nurse,label_photographer,label_physician,label_attorney,label_journalist,label_software_enginner,label_model
7,crawl-data/CC-MAIN-2016-44/segments/1476988720...,"Edmund J. Bourne, PhD, is a psychologist in no...","(Edmund, J, Bourne)",psychologist,M,136,psychologist,http://www.alibris.co.uk/search/books/author/E...,"_ is author of several books, including the be...",7,...,0,0,0,0,0,0,0,0,0,0
29,crawl-data/CC-MAIN-2016-44/segments/1476988718...,David Ellis is an attorney in serious need of ...,"(David, , Ellis)",attorney,M,60,attorney,http://www.sun-sentinel.com/ct-prj-0701-david-...,Not that _'s unprepared to face any judge or j...,29,...,0,0,0,0,0,0,0,0,0,0
32,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Xinsheng Chen is an architect at Cignex Techno...,"(Xinsheng, , Chen)",architect,M,58,architect,http://www.packtpub.com/liferay-user-interface...,_ holds a Master of Science degree in Computer...,32,...,0,0,0,0,0,0,0,0,0,0
66,crawl-data/CC-MAIN-2016-44/segments/1476988722...,"Suzanne B. Phillips, PsyD, ABPP, is a psycholo...","(Suzanne, B, Phillips)",psychologist,F,122,psychologist,http://idreambooks.com/Healing-Together-A-Coup...,_ is adjunct professor in the clinical doctora...,66,...,0,0,0,0,0,0,0,0,0,0
67,crawl-data/CC-MAIN-2017-34/segments/1502886105...,Komatsu Akina is a passionate photographer for...,"(Komatsu, , Akina)",photographer,F,77,photographer,http://goodmanga.net/5127/shinbunbu-no-komatsu...,When Aoi-kun of the Student Council confesses ...,67,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397422,crawl-data/CC-MAIN-2018-43/segments/1539583516...,"“Godo Röben is a role model for up-to-date, su...","(Godo, , Röben)",model,M,114,model,http://innovationstag.de/en/cmo-of-the-year/cm...,_ not only managed to make a local sausage fac...,397422,...,0,0,0,0,0,0,0,0,0,0
397466,crawl-data/CC-MAIN-2018-43/segments/1539583516...,Ryan Mounts is a Senior Software Engineer at S...,"(Ryan, , Mounts)",Senior Software Engineer,M,55,software_engineer,https://sideplate.com/culturecareers/our-team/,"_ has a Bachelor's degree from UC Berkeley, an...",397466,...,0,0,0,0,0,0,0,0,0,0
397492,crawl-data/CC-MAIN-2018-43/segments/1539583516...,Brian Friedkin is a passionate software engine...,"(Brian, , Friedkin)",software engineer,M,145,software_engineer,http://www.moddable.com/brian-friedkin,_ is particularly excited about portable softw...,397492,...,0,0,0,0,0,0,0,0,0,0
397764,crawl-data/CC-MAIN-2018-43/segments/1539583517...,Dr. Apostle Johnson is an exemplary model of t...,"(Apostle, , Johnson)",model,F,61,model,http://jesuswayfellowship.org/pastor.html,The power of God in _ draws and leaves a mark ...,397764,...,0,0,0,0,0,0,0,0,0,0


In [17]:
# only reviewed and valid samples as multi-label
df = filter_dataset(merged_df, classes=reviewed_classes, single_label=False, review_only=True, valid_only=True, remove_other_classes=False)
df

,path,raw,name,auto_raw_title,gender,start_pos,auto_title,URI,bio,index,...,label_dentist,label_teacher,label_psychologist,label_nurse,label_photographer,label_physician,label_attorney,label_journalist,label_software_enginner,label_model
32,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Xinsheng Chen is an architect at Cignex Techno...,"(Xinsheng, , Chen)",architect,M,58,architect,http://www.packtpub.com/liferay-user-interface...,_ holds a Master of Science degree in Computer...,32,...,0,0,0,0,0,0,0,0,0,0
66,crawl-data/CC-MAIN-2016-44/segments/1476988722...,"Suzanne B. Phillips, PsyD, ABPP, is a psycholo...","(Suzanne, B, Phillips)",psychologist,F,122,psychologist,http://idreambooks.com/Healing-Together-A-Coup...,_ is adjunct professor in the clinical doctora...,66,...,0,0,0,0,0,0,0,0,0,0
86,crawl-data/CC-MAIN-2017-13/segments/1490218195...,"Lee Beckstead, PhD, is a licensed psychologist...","(Lee, , Beckstead)",psychologist,M,265,psychologist,http://conferences.circlingthewagons.org/lee-b...,_ specializes in relationship issues such as i...,86,...,0,0,1,0,0,0,0,0,0,0
97,crawl-data/CC-MAIN-2016-44/segments/1476988720...,Jim Wark is an aerial photographer who special...,"(Jim, , Wark)",photographer,M,131,photographer,http://www.alibris.co.uk/search/books/author/J...,"Working alone, _ has photographed the continen...",97,...,0,0,0,0,1,0,0,0,0,0
103,crawl-data/CC-MAIN-2017-34/segments/1502886102...,Sally Jenkins is an award-winning journalist f...,"(Sally, , Jenkins)",journalist,F,141,journalist,http://howtogetalifenow.blogspot.com/2015/10/s...,In 2012 _ published the No.1 bestseller Sum it...,103,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397097,crawl-data/CC-MAIN-2018-43/segments/1539583516...,James Overly is a computer software engineer w...,"(James, , Overly)",software engineer,M,169,software_engineer,https://www.nasa.gov/50th/Folklife/biosSpaceSc...,"There are two main sets of software. First, th...",397097,...,0,0,0,0,0,0,0,0,0,0
397155,crawl-data/CC-MAIN-2018-43/segments/1539583516...,Mike Tricker is a lifelong model railroader. B...,"(Mike, , Tricker)",model,M,44,model,http://mrr.trains.com/how-to/model-train-layou...,"Based on British prototypes, _ previous layout...",397155,...,0,0,0,0,0,0,0,0,0,0
397466,crawl-data/CC-MAIN-2018-43/segments/1539583516...,Ryan Mounts is a Senior Software Engineer at S...,"(Ryan, , Mounts)",Senior Software Engineer,M,55,software_engineer,https://sideplate.com/culturecareers/our-team/,"_ has a Bachelor's degree from UC Berkeley, an...",397466,...,0,0,0,0,0,0,0,0,0,0
397492,crawl-data/CC-MAIN-2018-43/segments/1539583516...,Brian Friedkin is a passionate software engine...,"(Brian, , Friedkin)",software engineer,M,145,software_engineer,http://www.moddable.com/brian-friedkin,_ is particularly excited about portable softw...,397492,...,0,0,0,0,0,0,0,0,0,0


In [18]:
# only reviewed and valid samples as single-label, remove other classes
df = filter_dataset(merged_df, classes=reviewed_classes, single_label=True, review_only=True, valid_only=True, remove_other_classes=False)
df

,path,raw,name,auto_raw_title,gender,start_pos,auto_title,URI,bio,index,review,raw_titles,titles,valid,label
32,crawl-data/CC-MAIN-2013-20/segments/1368696381...,Xinsheng Chen is an architect at Cignex Techno...,"(Xinsheng, , Chen)",architect,M,58,architect,http://www.packtpub.com/liferay-user-interface...,_ holds a Master of Science degree in Computer...,32,1.0,[architect],[architect],1.0,0
66,crawl-data/CC-MAIN-2016-44/segments/1476988722...,"Suzanne B. Phillips, PsyD, ABPP, is a psycholo...","(Suzanne, B, Phillips)",psychologist,F,122,psychologist,http://idreambooks.com/Healing-Together-A-Coup...,_ is adjunct professor in the clinical doctora...,66,1.0,[psychologist/associate professor],[psychologist/associate professor],1.0,-1
86,crawl-data/CC-MAIN-2017-13/segments/1490218195...,"Lee Beckstead, PhD, is a licensed psychologist...","(Lee, , Beckstead)",psychologist,M,265,psychologist,http://conferences.circlingthewagons.org/lee-b...,_ specializes in relationship issues such as i...,86,1.0,[psychotherapist],[psychologist],1.0,4
97,crawl-data/CC-MAIN-2016-44/segments/1476988720...,Jim Wark is an aerial photographer who special...,"(Jim, , Wark)",photographer,M,131,photographer,http://www.alibris.co.uk/search/books/author/J...,"Working alone, _ has photographed the continen...",97,1.0,[photographer],[photographer],1.0,6
103,crawl-data/CC-MAIN-2017-34/segments/1502886102...,Sally Jenkins is an award-winning journalist f...,"(Sally, , Jenkins)",journalist,F,141,journalist,http://howtogetalifenow.blogspot.com/2015/10/s...,In 2012 _ published the No.1 bestseller Sum it...,103,1.0,[journalist],[journalist],1.0,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397097,crawl-data/CC-MAIN-2018-43/segments/1539583516...,James Overly is a computer software engineer w...,"(James, , Overly)",software engineer,M,169,software_engineer,https://www.nasa.gov/50th/Folklife/biosSpaceSc...,"There are two main sets of software. First, th...",397097,1.0,[software engineer],[software engineer],1.0,-1
397155,crawl-data/CC-MAIN-2018-43/segments/1539583516...,Mike Tricker is a lifelong model railroader. B...,"(Mike, , Tricker)",model,M,44,model,http://mrr.trains.com/how-to/model-train-layou...,"Based on British prototypes, _ previous layout...",397155,1.0,[model railroader],[*engineer],1.0,-1
397466,crawl-data/CC-MAIN-2018-43/segments/1539583516...,Ryan Mounts is a Senior Software Engineer at S...,"(Ryan, , Mounts)",Senior Software Engineer,M,55,software_engineer,https://sideplate.com/culturecareers/our-team/,"_ has a Bachelor's degree from UC Berkeley, an...",397466,1.0,[senior software engineer],[software engineer],1.0,-1
397492,crawl-data/CC-MAIN-2018-43/segments/1539583516...,Brian Friedkin is a passionate software engine...,"(Brian, , Friedkin)",software engineer,M,145,software_engineer,http://www.moddable.com/brian-friedkin,_ is particularly excited about portable softw...,397492,1.0,[software engineer],[software engineer],1.0,-1
